<a href="https://colab.research.google.com/github/adhithya2525/adhi-15day-workshop/blob/main/Day4/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_bronze=spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/content/large_sales_data.csv")
print(f'Bronze table row count: {df_bronze.count()}')
print(f'Bronze table column count: {len(df_bronze.columns)}')

Bronze table row count: 5000
Bronze table column count: 13


In [ ]:
df_bronze.select('quantity','unit_price','revenue').describe().show()

+-------+-----------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|
|    min|                1|               600|               600|
|    max|               15|             45000|            675000|
+-------+-----------------+------------------+------------------+



In [ ]:
df_bronze.write \
    .mode('overwrite') \
    .parquet('sales_bronze.parquet')

In [ ]:
import os
def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total=0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      fp=os.path.join(dirpath,f)
      total+=os.path.getsize(fp)/1024
  return total

csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=round(csv_size/parquet_size,2)
print(f'CSV file size: {csv_size} KB')
print(f'Parquet file size: {parquet_size} KB')

CSV file size: 529.3125 KB
Parquet file size: 55.09765625 KB


In [ ]:
df_bronze.show(5,truncate=False)
df_bronze.select('quantity','unit_price','revenue').describe().show()

+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Kumar |Cash on D

In [ ]:
df_bronze_with_sum = df_bronze.withColumn('unit_price_revenue_sum', F.col('unit_price') + F.col('revenue'))

print('Schema with new column:')
df_bronze_with_sum.printSchema()

print('\nDataFrame with new column:')
df_bronze_with_sum.show(5, truncate=False)

Schema with new column:
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- unit_price_revenue_sum: integer (nullable = true)


DataFrame with new column:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+----------------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|unit_price_revenue_sum|
+--------+-----

In [ ]:
df_high_revenue = df_bronze.filter(col("revenue")>100000).show()


+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|
|    1002| Ramesh Kumar|Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|     Credit Card|     Shipped|
|    1004|   Suresh Rao| Tablet|Electronics|       5|     32000| 160000|2023-01-04|    Surat|  West|  Ravi Kumar|Cash on Delivery|  Processing|
|    1008|  Priya Patel| Laptop|Electronics|      13|     45000| 585000|2023-05-29|  Chennai| South| Meera Patel|     Credit Card|   Can

In [ ]:
df_sliver=df_bronze \
    .dropDuplicates() \
    .dropna(subset=['quantity','unit_price','revenue'])
df_sliver=df_sliver.withColumn("order_date",to_date(col("order_date"),"yyyy-MM-dd")) \
    .withColumn("year",year(col("order_date"))) \
    .withColumn("month",month(col("order_date")))
df_sliver=df_sliver.withColumn("revenue_category",F.when(col('revenue')>40000,'High')\
                                        .when((col('revenue')>10000) & (col('revenue')<=40000),'Medium')\
                                        .otherwise('Low'))
df_sliver.show(5)
print(f'Sliver table row count: {df_sliver.count()}')
print(f'Sliver table column count: {len(df_sliver.columns)}')

+--------+-------------+--------+-----------+--------+----------+-------+----------+---------+------+-----------+--------------+------------+----+-----+----------------+
|order_id|customer_name| product|   category|quantity|unit_price|revenue|order_date|     city|region|  sales_rep|payment_method|order_status|year|month|revenue_category|
+--------+-------------+--------+-----------+--------+----------+-------+----------+---------+------+-----------+--------------+------------+----+-----+----------------+
|    1246|  Divya Singh|Keyboard|Accessories|      11|      1200|  13200|2023-02-07|  Kolkata|  East|Kavya Reddy|   Net Banking|     Shipped|2023|    2|          Medium|
|    1323|   Ananya Das|  Webcam|Accessories|       7|      2500|  17500|2023-01-24|  Kolkata|  East|Meera Patel|           UPI|   Delivered|2023|    1|          Medium|
|    1386|Kavya Nambiar| Speaker|Electronics|      13|      4500|  58500|2023-04-16|Bangalore| South|Meera Patel|   Credit Card|  Processing|2023|    

In [ ]:
df_sliver.write \
    .mode('overwrite') \
    .parquet('sales_silver.parquet')

In [ ]:
df_verify=spark.read.parquet('sales_silver.parquet')
df_verify.show(5)
print(get_dir_size('sales_silver.parquet'))

+--------+-------------+--------+-----------+--------+----------+-------+----------+---------+------+-----------+--------------+------------+----+-----+----------------+
|order_id|customer_name| product|   category|quantity|unit_price|revenue|order_date|     city|region|  sales_rep|payment_method|order_status|year|month|revenue_category|
+--------+-------------+--------+-----------+--------+----------+-------+----------+---------+------+-----------+--------------+------------+----+-----+----------------+
|    1246|  Divya Singh|Keyboard|Accessories|      11|      1200|  13200|2023-02-07|  Kolkata|  East|Kavya Reddy|   Net Banking|     Shipped|2023|    2|          Medium|
|    1323|   Ananya Das|  Webcam|Accessories|       7|      2500|  17500|2023-01-24|  Kolkata|  East|Meera Patel|           UPI|   Delivered|2023|    1|          Medium|
|    1386|Kavya Nambiar| Speaker|Electronics|      13|      4500|  58500|2023-04-16|Bangalore| South|Meera Patel|   Credit Card|  Processing|2023|    

In [ ]:
df_verify.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

